# SE-ResNet-64 (Full Attention) — Kaggle GPU Run

**Architecture:** Original paper ResNet (64 block) + Squeeze-and-Excitation channel attention per block  
**Platform:** Kaggle GPU (T4/P100)  
**Goal:** Compare SE-ResNet vs base ResNet — same protocol, same metric  

### কীভাবে চালাবে
1. Kaggle-এ **'Add Data'** → DL_DOA_CLONE repo attach করো (pretrained `inf_model_007_256_resnet.h5` থাকলে warm-start হবে)
2. **GPU T4 x2** accelerator on করো
3. **'Run All'** দাও — বাকি সব auto হবে
4. Output `/kaggle/working/` এ save হবে

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

ensure('opencv-python-headless', 'cv2')
print('✅ Dependencies OK')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────
import os, math, time, random as pyrandom
import numpy as np
import scipy.ndimage
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.layers import (
    Conv2D, Input, BatchNormalization, Activation,
    Add, Conv2DTranspose, GlobalAveragePooling2D,
    Reshape, Multiply, Dense
)
from tensorflow.keras.models import Model
from scipy.optimize import linear_sum_assignment

tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42); pyrandom.seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    strategy = tf.distribute.MirroredStrategy()
    print(f'MirroredStrategy: {strategy.num_replicas_in_sync} GPU(s)')
else:
    strategy = tf.distribute.get_strategy()
    print('⚠️  No GPU found — CPU-only, training will be very slow')

In [ ]:
# ── Cell 3: Physics + Data Generator (inline, no repo dependency) ─────
class _H(np.ndarray):
    @property
    def H(self): return self.conj().transpose()

def ev(n, angle):
    return ((1/np.sqrt(n)) * np.exp(-1j*np.pi*np.cos(angle)*np.arange(n))).reshape(-1,1)

def make_F(P, nt):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(1j*(2*np.pi/P)*np.arange(P))))
    F = np.zeros((nt,P), dtype=complex)
    for i,ph in enumerate(phi): F[:,i] = ev(nt,ph).ravel()
    return F

def make_W(Q, nr):
    phi = np.arccos((1/np.pi)*np.angle(np.exp(-1j*(2*np.pi/Q)*np.arange(Q))))
    W = np.zeros((nr,Q), dtype=complex)
    for i,ph in enumerate(phi): W[:,i] = ev(nr,ph).ravel()
    return W

def gen_channel(nr, nt, phi_l, psi_l, alpha_l):
    H = np.zeros((nr,nt), dtype=complex)
    for a,phi,psi in zip(alpha_l,phi_l,psi_l):
        H += a * (ev(nr,psi) * ev(nt,phi).view(_H).H)
    return np.sqrt(nt*nr) * H

def gen_points(L, delta=np.pi/6, max_try=20000):
    pts = []
    for _ in range(max_try):
        if len(pts)==L: break
        x,y = np.random.uniform(0,np.pi), np.random.uniform(0,np.pi)
        if all(math.hypot(x-p[0],y-p[1])>=delta for p in pts):
            pts.append((x,y))
    if len(pts)<L: raise RuntimeError(f'Cannot place {L} points with delta={delta:.3f}')
    return pts

def gen_gt(phi_l, psi_l, M=256, sigma=0.07):
    op = np.mod(np.pi*np.cos(phi_l), 2*np.pi)
    oq = np.mod(-np.pi*np.cos(psi_l), 2*np.pi)
    margin = 3*sigma
    ax = np.linspace(-margin, 2*np.pi+margin, M, endpoint=False)
    Wp,Wq = np.meshgrid(ax,ax)
    coeff = 1/(2*np.pi*sigma**2)
    G = sum(coeff*np.exp(-((Wp-o)**2+(Wq-p)**2)/(2*sigma**2)) for o,p in zip(op,oq))
    return G.astype(np.float32)

def data_generation(Training=True, condition=None, sigma=0.07, M=256):
    """Infinite generator — paper-exact distribution."""
    while True:
        if Training:
            L   = np.random.randint(1, 10)
            SNR = np.random.randint(-15, 25)
            P   = pyrandom.choice([16, 32])
            nt  = 16 if P==16 else pyrandom.choice([16,32])
        else:
            L, SNR, P, nt = condition
        Q = nr = P, nt
        Q, nr = P, nt
        F = make_F(P, nt); W = make_W(Q, nr)
        alpha = (np.sqrt(1/L)/np.sqrt(2))*(np.random.randn(L)+1j*np.random.randn(L))
        alpha = alpha[np.argsort(-np.abs(alpha))]
        pts   = gen_points(L)
        phi_l = [p[0] for p in pts]; psi_l = [p[1] for p in pts]
        H = gen_channel(nr, nt, phi_l, psi_l, alpha)
        var = 10**(-SNR/10); s = np.sqrt(var/2)
        Z = s*(np.random.randn(Q,P)+1j*np.random.randn(Q,P))
        Y = H if False else (W.view(_H).H @ H) @ F + Z
        zoom = 4 if P==16 else 2
        data = np.stack([scipy.ndimage.zoom(Y.real,zoom,order=0),
                         scipy.ndimage.zoom(Y.imag,zoom,order=0)], axis=-1).astype(np.float32)
        if Training:
            gt = gen_gt(phi_l,psi_l,M,sigma)[...,np.newaxis]
            yield data, gt
        else:
            yield data, np.stack([psi_l,phi_l]).astype(np.float32)

print('✅ Physics + data generator ready')

In [ ]:
# ── Cell 4: Architecture — SE-ResNet-64 ──────────────────────────────
def se_block(x, filters, ratio=4):
    """Squeeze-and-Excitation channel attention gate."""
    s = GlobalAveragePooling2D()(x)
    s = Dense(max(filters//ratio, 1), activation='relu')(s)
    s = Dense(filters, activation='sigmoid')(s)
    s = Reshape((1, 1, filters))(s)
    return Multiply()([x, s])

def se_res_conv(x, filters=12, se_ratio=4):
    """Residual block + SE attention (paper block + SE gate before Add)."""
    skip = x
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 5, padding='same')(x)
    x = BatchNormalization()(x)
    x = se_block(x, filters, se_ratio)   # ← SE attention
    x = Add()([x, skip])
    x = Activation('relu')(x)
    return x

def build_SE_ResNet(n_blocks=64, filters=12, se_ratio=4):
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, 5, strides=2, padding='same')(x_in)
    for _ in range(n_blocks):
        x = se_res_conv(x, filters, se_ratio)
    x = Conv2DTranspose(1, 5, strides=2, padding='same')(x)
    return Model(x_in, x, name=f'SE-ResNet-{n_blocks}b')

def build_plain_ResNet(n_blocks=64, filters=12):
    """Base paper ResNet (for warm-start weight transfer)."""
    def res_conv(x, f):
        skip = x
        x = Conv2D(f,5,padding='same')(x); x=BatchNormalization()(x); x=Activation('relu')(x)
        x = Conv2D(f,5,padding='same')(x); x=BatchNormalization()(x)
        x = Add()([x,skip]); x=Activation('relu')(x)
        return x
    x_in = Input(shape=(64,64,2))
    x = Conv2DTranspose(filters,5,strides=2,padding='same')(x_in)
    for _ in range(n_blocks): x = res_conv(x, filters)
    x = Conv2DTranspose(1,5,strides=2,padding='same')(x)
    return Model(x_in, x, name='ResNet-64b')

# param count দেখো
m_se  = build_SE_ResNet()
m_ref = build_plain_ResNet()
print(f'SE-ResNet-64:   {m_se.count_params():,} params')
print(f'Plain ResNet-64: {m_ref.count_params():,} params  (reference: 469,393)')
overhead = (m_se.count_params()/m_ref.count_params() - 1)*100
print(f'SE overhead:    +{overhead:.2f}%')
del m_ref

In [ ]:
# ── Cell 5: Warm-start from pretrained ResNet (auto) ─────────────────
def find_pretrained(filename='inf_model_007_256_resnet.h5'):
    roots = ['/kaggle/input', '/kaggle/working', '/content', '.']
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, files in os.walk(root):
            if filename in files: return os.path.join(dp, filename)
    return None

def warm_start(se_model, pretrained_path, n_blocks=64):
    """Conv/BN/ConvTranspose weight গুলো pretrained ResNet থেকে copy করো."""
    orig = build_plain_ResNet(n_blocks)
    orig.load_weights(pretrained_path)
    orig_w = [l for l in orig.layers if l.get_weights()]
    se_w   = [l for l in se_model.layers
               if l.get_weights() and not isinstance(l, Dense)]
    assert len(orig_w)==len(se_w), f'Layer mismatch: {len(orig_w)} vs {len(se_w)}'
    for lo,ls in zip(orig_w, se_w): ls.set_weights(lo.get_weights())
    n_dense = sum(1 for l in se_model.layers if isinstance(l, Dense))
    print(f'✅ Warm-start: {len(orig_w)} layers transferred from pretrained ResNet')
    print(f'   {n_dense} SE-gate Dense layers remain randomly initialized')
    del orig
    return se_model

PRETRAINED_PATH = find_pretrained()
WARM = PRETRAINED_PATH is not None
if WARM:
    print(f'Pretrained weights found: {PRETRAINED_PATH}')
else:
    print('Pretrained weights not found — will train from scratch')
    print('(Kaggle-এ DL_DOA_CLONE dataset attach করলে warm-start পাবে)')

In [ ]:
# ── Cell 6: Training config ───────────────────────────────────────────
#
# Warm-start  → 50 epoch fine-tune  (SE gate শিখবে, backbone already trained)
# Scratch     → 500 epoch           (paper scale)
#
BATCH           = 32     # paper Table I
STEPS_PER_EPOCH = 312    # 10000 // 32  (paper: 10000 samples/epoch)
EPOCHS          = 50  if WARM else 500
LR              = 1e-4 if WARM else 1e-3
N_BLOCKS        = 64
SE_RATIO        = 4
OUT_DIR         = '/kaggle/working'
WEIGHTS_PATH    = os.path.join(OUT_DIR, 'se_resnet64_best.weights.h5')

print(f'Config:  BATCH={BATCH}  STEPS={STEPS_PER_EPOCH}  EPOCHS={EPOCHS}  LR={LR}')
print(f'WARM_START={WARM}  |  Total samples this run: {BATCH*STEPS_PER_EPOCH*EPOCHS:,}')

In [ ]:
# ── Cell 7: Build model + compile ─────────────────────────────────────
with strategy.scope():
    model = build_SE_ResNet(n_blocks=N_BLOCKS, se_ratio=SE_RATIO)
    if WARM:
        model = warm_start(model, PRETRAINED_PATH, n_blocks=N_BLOCKS)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss='mse'
    )
model.summary(line_length=80)

In [ ]:
# ── Cell 8: Training ──────────────────────────────────────────────────
def make_train_ds(batch):
    gen = data_generation(Training=True)
    def fn():
        for d,g in gen: yield d, g
    ds = tf.data.Dataset.from_generator(
        fn,
        output_signature=(
            tf.TensorSpec(shape=(64,64,2), dtype=tf.float32),
            tf.TensorSpec(shape=(256,256,1), dtype=tf.float32),
        )
    )
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        WEIGHTS_PATH, save_best_only=True,
        save_weights_only=True, monitor='loss', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='loss', factor=0.5, patience=10,
        min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(OUT_DIR, 'train_log.csv')),
]

train_ds = make_train_ds(BATCH)

print(f'Training SE-ResNet-{N_BLOCKS}b  |  {EPOCHS} epochs × {STEPS_PER_EPOCH} steps × {BATCH} batch')
print(f'Weights will be saved to: {WEIGHTS_PATH}')
print('─'*60)

t0 = time.time()
history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
elapsed = time.time() - t0
print(f'\n✅ Training done in {elapsed/3600:.2f}h  |  best loss: {min(history.history["loss"]):.6f}')

# best weights load করো
model.load_weights(WEIGHTS_PATH)
print('Best weights loaded for evaluation.')

In [ ]:
# ── Cell 9: Training curve ────────────────────────────────────────────
plt.figure(figsize=(9, 3.5))
plt.plot(history.history['loss'], color='crimson', linewidth=1.5, label='train MSE loss')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('SE-ResNet-64 Training Loss')
plt.grid(alpha=.3); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'training_curve.png'), dpi=150)
plt.show()
print('Saved: training_curve.png')

In [ ]:
# ── Cell 10: Evaluation utilities ─────────────────────────────────────
def get_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor=True; p.blobColor=255
    p.minThreshold=0; p.maxThreshold=255
    p.filterByArea=True; p.minArea=1; p.maxArea=1000
    p.filterByCircularity=p.filterByConvexity=p.filterByInertia=False
    return cv2.SimpleBlobDetector_create(p)

DETECTOR = get_detector()

def get_peaks(pred2d, L):
    img = cv2.normalize(pred2d, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    kps = DETECTOR.detect(img)
    if not kps: return np.zeros((0,2))
    coords = np.array([k.pt for k in kps])
    amps = np.array([img[int(round(k.pt[1])), int(round(k.pt[0]))] for k in kps])
    return coords[np.argsort(-amps)[:L]]

def peaks2angles(peaks, sigma=0.07, M=256):
    if len(peaks)==0: return np.array([]), np.array([])
    margin=3*sigma; ext=2*np.pi+2*margin
    f = np.where(
        (-margin + peaks.T/M*ext) > np.pi,
        (-margin + peaks.T/M*ext) - 2*np.pi,
        (-margin + peaks.T/M*ext)
    )
    psi = np.arccos(np.clip(-f[1]/np.pi, -1, 1))
    phi = np.arccos(np.clip( f[0]/np.pi, -1, 1))
    return psi, phi

def match_angles(est, feat):
    L = feat.shape[-1]
    if len(est[0]) < L:
        return feat, np.full((2,L), np.nan)
    est = (est[0][:L], est[1][:L])
    A = np.stack(feat.T);   B = np.array(list(zip(est[0],est[1])))
    A2 = np.array(list(zip(feat[0],feat[1])))
    dist = np.linalg.norm(A2[:,None]-B[None], axis=2)
    r,c = linear_sum_assignment(dist)
    gt_m  = np.array([[feat[0][i],feat[1][i]] for i in r]).T
    est_m = np.array([[est[0][j],est[1][j]] for j in c]).T
    return gt_m, est_m

def ang_diff_deg(gt, pred):
    return (np.angle(np.exp(1j*gt)*np.exp(-1j*pred))*180/np.pi).flatten()

print('✅ Evaluation utilities ready')

In [ ]:
# ── Cell 11: Full evaluation (paper-scale: L=3, 8 SNR × 1000 samples) ─
SNRS_EVAL    = list(range(-10, 30, 5))   # -10,-5,0,5,10,15,20,25
N_PER_SNR    = 1000
MAX_DEG_ERR  = 1.0

se_rmse = {}; se_pd = {}

for snr in SNRS_EVAL:
    gen = data_generation(Training=False, condition=(3, snr, 16, 16))
    good_all = []; n_total = 0

    for i in range(N_PER_SNR):
        data, feat = next(gen)
        pred = model(tf.expand_dims(data,0), training=False)[0,:,:,0].numpy()
        peaks = get_peaks(pred, L=3)
        est   = peaks2angles(peaks)
        gt_m, pr_m = match_angles(est, feat)
        n_total += gt_m.shape[1]
        if np.isnan(pr_m).any(): continue
        diffs = ang_diff_deg(gt_m, pr_m)
        good_all.extend(diffs[np.abs(diffs) <= MAX_DEG_ERR])

    good_all = np.array(good_all)
    se_pd[snr]   = len(good_all) / n_total if n_total else np.nan
    se_rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) else np.nan
    print(f'SNR={snr:4d}dB  Pd={se_pd[snr]:.4f}  RMSE={se_rmse[snr]:.4f}')

print('\n✅ Evaluation done')

In [ ]:
# ── Cell 12: Comparison table vs base ResNet ──────────────────────────
REF_RMSE = {-10:0.5532,-5:0.5118, 0:0.4581, 5:0.3920,
             10:0.3256,15:0.2792,20:0.2528,25:0.2377}
REF_PD   = {-10:0.2043,-5:0.4366, 0:0.6396, 5:0.7790,
             10:0.8623,15:0.8987,20:0.9250,25:0.9378}

print('='*72)
print(f'{"SNR":>5} | {"SE-ResNet RMSE":>14} {"SE-ResNet Pd":>12} | {"ResNet RMSE":>11} {"ResNet Pd":>9} | {"ΔPd":>7}')
print('-'*72)
for s in SNRS_EVAL:
    dpd = se_pd[s] - REF_PD[s]
    print(f'{s:>5} | {se_rmse[s]:>14.4f} {se_pd[s]:>12.4f} | '
          f'{REF_RMSE[s]:>11.4f} {REF_PD[s]:>9.4f} | {dpd:>+7.4f}')
print('='*72)

mean_dpd = np.nanmean([se_pd[s]-REF_PD[s] for s in SNRS_EVAL])
low_snr  = [-10,-5,0,5]
mean_low = np.nanmean([se_pd[s]-REF_PD[s] for s in low_snr])
print(f'\nMean ΔPd (all SNR):        {mean_dpd:+.4f}')
print(f'Mean ΔPd (low SNR only):   {mean_low:+.4f}  ← attention এখানে help করে কিনা')
print(f'Params: SE-ResNet={model.count_params():,}  vs  ResNet=469,393')

In [ ]:
# ── Cell 13: Result plots ─────────────────────────────────────────────
snrs = sorted(se_rmse)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

axs[0].plot(snrs, [se_rmse[s] for s in snrs], 'o-', color='crimson',
            linewidth=2, markersize=6, label='SE-ResNet-64 (ours)')
axs[0].plot(snrs, [REF_RMSE[s] for s in snrs], 's--', color='steelblue',
            linewidth=2, markersize=6, label='ResNet (paper baseline)')
axs[0].axhline(1/np.sqrt(3), ls=':', c='gray', label='random floor')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)')
axs[0].set_title('RMSE vs SNR  (L=3, P=16)')
axs[0].legend(); axs[0].grid(alpha=.3)

axs[1].plot(snrs, [se_pd[s] for s in snrs], 'o-', color='crimson',
            linewidth=2, markersize=6, label='SE-ResNet-64 (ours)')
axs[1].plot(snrs, [REF_PD[s] for s in snrs], 's--', color='steelblue',
            linewidth=2, markersize=6, label='ResNet (paper baseline)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd')
axs[1].set_ylim(-0.02, 1.05)
axs[1].set_title('Detection Probability vs SNR  (L=3, P=16)')
axs[1].legend(); axs[1].grid(alpha=.3)

plt.suptitle(f'SE-ResNet-64 vs ResNet  |  params: {model.count_params():,} vs 469,393',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'SE_ResNet_vs_ResNet.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: SE_ResNet_vs_ResNet.png')

In [ ]:
# ── Cell 14: Save results ─────────────────────────────────────────────
import json

results = {
    'model'    : f'SE-ResNet-{N_BLOCKS}b',
    'params'   : model.count_params(),
    'warm_start': WARM,
    'epochs'   : EPOCHS,
    'se_rmse'  : {str(k): float(v) for k,v in se_rmse.items()},
    'se_pd'    : {str(k): float(v) for k,v in se_pd.items()},
    'ref_rmse' : {str(k): float(v) for k,v in REF_RMSE.items()},
    'ref_pd'   : {str(k): float(v) for k,v in REF_PD.items()},
}

json_path = os.path.join(OUT_DIR, 'se_resnet64_results.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)

print('✅ Results saved:')
print(f'  {WEIGHTS_PATH}')
print(f'  {json_path}')
print(f'  {OUT_DIR}/SE_ResNet_vs_ResNet.png')
print(f'  {OUT_DIR}/training_curve.png')
print(f'  {OUT_DIR}/train_log.csv')
print('\n/kaggle/working থেকে সব download করো।')